In [ ]:
import torch
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")


login(token=HF_TOKEN)

print("PyTorch version:", torch.__version__)
print("CUDA Available", torch.cuda.is_available())
if torch.cuda.is_available():
  print("Device name", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.0 MB/s eta 0:00:00
PyTorch version: 2.11.0+cu128
CUDA Available True
Device name Tesla T4


In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MODEL_CKPT = "Davlan/afro-xlmr-base"
FOLDER_PATH = "../data/processed"

train_df = pd.read_csv(os.path.join(FOLDER_PATH, "train.csv"))
val_df = pd.read_csv(os.path.join(FOLDER_PATH, "val.csv"))
test_df = pd.read_csv(os.path.join(FOLDER_PATH, "test.csv"))

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT, token=HF_TOKEN)

print("Successfully fetched tokenizer")

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Successfully fetched tokenizer


In [ ]:
def tokenizer_batch(example):
    return tokenizer(example["tweet"], truncate=True, max_length=128)


tokenized_train = train_dataset.map(tokenizer_batch, batched=True)
tokenized_val = val_dataset.map(tokenizer_batch, batched=True)
tokenized_test = test_dataset.map(tokenizer_batch, batched=True)

id2label = {0: "Positive", 1: "Neutral", 2: "Negative"}
label2id = {"Positive": 0, "Neutral": 1, "Negative": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT, num_labels=3, id2label=id2label, label2id=label2id, token=HF_TOKEN
)

Map:   0%|          | 0/6041 [00:00<?, ? examples/s]

Map:   0%|          | 0/1295 [00:00<?, ? examples/s]

Map:   0%|          | 0/1295 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro"
    )
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "macro_f1": f1, "precision": precision, "recall": recall}


training_args = TrainingArguments(
    output_dir="./afro-xlmr-baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    seed=45,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.save_model("./afro-xlmr-baseline")
tokenizer.save_pretrained("./afro-xlmr-baseline")

print("Training complete and best model saved to ./afro-xlmr-baseline")

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision,Recall
1,0.233330,1.360759,0.658687,0.483427,0.600627,0.472874
2,0.242513,1.448415,0.656371,0.524572,0.542879,0.515398
3,0.243296,1.606727,0.667954,0.523186,0.541928,0.511907
4,0.375880,1.544167,0.659459,0.523669,0.537550,0.514533


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and best model saved to ./afro-xlmr-best
